---
# ─── RACE PLANNING ───────────────────────────────────────────────
---

### GAP — Grade-Adjusted Pace per segment
Normalises pace to flat-equivalent effort. Uses Minetti's metabolic cost formula.

In [ ]:
def minetti_cost(grade_frac):
    """Metabolic cost relative to flat running (Minetti 2002).
    grade_frac: slope as fraction (e.g. 0.10 for 10%)
    Returns cost ratio vs flat.
    """
    g = np.clip(grade_frac, -0.45, 0.45)
    cost = (155.4*g**5 - 30.4*g**4 - 43.3*g**3 + 46.3*g**2 + 19.5*g + 3.6)
    return cost / 3.6   # normalised to flat (flat cost ≈ 3.6)

for seg in segments:
    sub = seg['data'].copy()
    ddist = sub['dist_km'].diff() * 1000          # metres
    dalt  = sub['alt'].diff()
    grade_frac = (dalt / ddist.replace(0, np.nan)).clip(-0.45, 0.45)
    cost_ratio = grade_frac.apply(minetti_cost)
    # GAP = actual_pace / cost_ratio  (lower ratio on downhill = slower GAP)
    gap = sub['pace'] / cost_ratio.replace(0, np.nan)
    seg['gap_median']  = gap.median()
    seg['gap_mean']    = gap.mean()
    seg['pace_median'] = sub['pace'].median()

# ── Plot ──────────────────────────────────────────────────────────
names     = [s['name'].split('–')[-1].strip() for s in segments]
pace_vals = [s['pace_median'] for s in segments]
gap_vals  = [s['gap_median']  for s in segments]

x = np.arange(len(segments))
w = 0.35
fig, ax = plt.subplots(figsize=(max(8, len(segments)*1.6), 4))
ax.bar(x - w/2, pace_vals, w, label='Actual pace',     color='steelblue',  alpha=0.85)
ax.bar(x + w/2, gap_vals,  w, label='GAP (flat equiv)', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Pace [min/km]')
ax.invert_yaxis()   # lower = faster
ax.set_title('Actual pace vs Grade-Adjusted Pace per segment')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('gap_per_segment.png', dpi=150, bbox_inches='tight'); plt.show()

print('\nGAP summary:')
for s in segments:
    print(f"  {s['name']:40s}  actual {s['pace_median']:.2f}  GAP {s['gap_median']:.2f} min/km")

### Optimal pacing strategy — even-effort race plan
Given a target finish time, back-calculates the GAP you need to hold, then converts back
to actual required pace for each segment using the slope profile.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────
TARGET_FINISH_MIN = 90.0    # your goal finish time in minutes
# ──────────────────────────────────────────────────────────────

# Flat-equivalent total distance
total_flat_km = sum(s['gap_median'] and s['dist_km'] * (s['pace_median'] / s['gap_median'])
                    for s in segments if s.get('gap_median'))
target_gap = TARGET_FINISH_MIN / total_flat_km  # min/km on flat equivalent

plan = []
for s in segments:
    if not s.get('gap_median') or s['gap_median'] == 0: continue
    ratio        = s['pace_median'] / s['gap_median']   # actual/GAP ratio from training
    target_pace  = target_gap * ratio                   # required actual pace
    target_time  = target_pace * s['dist_km']           # minutes
    faster_pct   = (s['pace_median'] - target_pace) / s['pace_median'] * 100
    plan.append({
        'segment':      s['name'].split('–')[-1].strip(),
        'dist_km':      s['dist_km'],
        'training_pace': s['pace_median'],
        'target_pace':  target_pace,
        'target_time_min': target_time,
        'delta_pct':    faster_pct,
    })

plan_df = pd.DataFrame(plan)
plan_df['cumulative_time'] = plan_df['target_time_min'].cumsum()

# ── Plot ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(8, len(plan)*1.6), 7), sharex=True)

x = np.arange(len(plan_df))
ax1.bar(x, plan_df['training_pace'], 0.4, label='Training pace', color='steelblue', alpha=0.7)
ax1.bar(x + 0.4, plan_df['target_pace'], 0.4, label=f'Target pace ({TARGET_FINISH_MIN:.0f} min)', color='tomato', alpha=0.85)
ax1.set_ylabel('Pace [min/km]'); ax1.invert_yaxis()
ax1.set_title(f'Race plan — target {TARGET_FINISH_MIN:.0f} min finish')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

colors = ['tomato' if d < 0 else 'seagreen' for d in plan_df['delta_pct']]
ax2.bar(x + 0.2, plan_df['delta_pct'], 0.6, color=colors, alpha=0.8)
ax2.axhline(0, color='k', lw=0.8)
ax2.set_ylabel('Pace change vs training [%]\n(+) = need to go faster')
ax2.set_xticks(x + 0.2); ax2.set_xticklabels(plan_df['segment'], rotation=20, ha='right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.savefig('race_plan.png', dpi=150, bbox_inches='tight'); plt.show()

print(f'\nTarget GAP to hold: {target_gap:.2f} min/km (flat equivalent)\n')
print(f'{"Segment":<35} {"Dist":>6} {"Train":>7} {"Target":>7} {"Time":>7} {"Cumul":>7} {"Δ%":>6}')
print('-'*75)
for _, r in plan_df.iterrows():
    print(f"{r['segment']:<35} {r['dist_km']:>5.2f}k  {r['training_pace']:>5.2f}  {r['target_pace']:>5.2f}  "
          f"{r['target_time_min']:>5.1f}m  {r['cumulative_time']:>5.1f}m  {r['delta_pct']:>+5.1f}%")
print(f"\nTotal projected: {plan_df['target_time_min'].sum():.1f} min")

### HR drift — fatigue accumulation across segments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: HR vs GAP scatter per segment
for seg in segments:
    sub = seg['data']
    if 'hr' not in sub.columns: continue
    color = TERRAIN_COLORS.get(seg['terrain'], DEFAULT_COLOR)
    axes[0].scatter([seg['gap_median']], [sub['hr'].mean()],
                    color=color, s=120, zorder=5,
                    label=seg['name'].split('–')[-1].strip())
axes[0].set_xlabel('GAP [min/km]'); axes[0].set_ylabel('Avg HR [bpm]')
axes[0].set_title('Aerobic efficiency: HR vs GAP per segment')
axes[0].invert_xaxis()  # faster pace on right
axes[0].legend(fontsize=7, frameon=False)
axes[0].grid(alpha=0.3)

# Right: HR over cumulative distance (full run), segment boundaries marked
offset = 0.0
for seg in segments:
    sub = seg['data']
    if 'hr' not in sub.columns: continue
    color = TERRAIN_COLORS.get(seg['terrain'], DEFAULT_COLOR)
    x = sub['rel_dist'] + offset
    hr_smooth = sub['hr'].rolling(30, center=True).mean()
    axes[1].plot(x, hr_smooth, color=color, lw=2)
    axes[1].axvline(offset, color='k', lw=0.6, ls='--', alpha=0.4)
    offset += seg['dist_km']

axes[1].set_xlabel('Cumulative dist [km]'); axes[1].set_ylabel('HR [bpm]')
axes[1].set_title('HR evolution (coloured by segment/terrain)')
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('hr_drift.png', dpi=150, bbox_inches='tight'); plt.show()

---
# ─── PERFORMANCE CHARACTERISATION ───────────────────────────────
---

### Power efficiency per segment — watts per vertical metre

In [ ]:
eff_rows = []
for seg in segments:
    sub = seg['data']
    if 'power' not in sub.columns or 'alt' not in sub.columns: continue
    gain = sub['alt'].diff().clip(lower=0).sum()
    avg_power = sub['power'].replace(0, np.nan).median()
    # W / (m of vertical per second) — lower = more efficient climber
    duration_s = (sub['timestamp'].iloc[-1] - sub['timestamp'].iloc[0]).total_seconds()
    vspeed_mps = gain / duration_s if duration_s > 0 else np.nan   # vertical m/s
    w_per_vm   = avg_power / vspeed_mps if vspeed_mps and vspeed_mps > 0 else np.nan
    # Also: speed / power = mechanical efficiency
    spd = sub['speed'].replace(0, np.nan).median() if 'speed' in sub else np.nan
    mech_eff = (spd / avg_power * 1000) if avg_power and spd else np.nan
    eff_rows.append({
        'segment':       seg['name'].split('–')[-1].strip(),
        'terrain':       seg['terrain'],
        'avg_power_W':   avg_power,
        'vspeed_mpm':    vspeed_mps * 60 if vspeed_mps else np.nan,  # m/min
        'W_per_Vm':      w_per_vm,
        'mech_eff':      mech_eff,
    })

eff_df = pd.DataFrame(eff_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = [TERRAIN_COLORS.get(r['terrain'], DEFAULT_COLOR) for _, r in eff_df.iterrows()]

for ax, col, title, ylabel in zip(
    axes,
    ['avg_power_W', 'vspeed_mpm', 'W_per_Vm'],
    ['Average power', 'Vertical speed', 'Power cost of climbing'],
    ['Watts', 'm/min', 'W / (m/s vert)']):
    vals = eff_df[col].values
    ax.bar(range(len(eff_df)), vals, color=colors, alpha=0.85, edgecolor='white')
    ax.set_xticks(range(len(eff_df)))
    ax.set_xticklabels(eff_df['segment'], rotation=25, ha='right', fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.savefig('power_efficiency.png', dpi=150, bbox_inches='tight'); plt.show()
display(eff_df)

### Cadence & step length vs slope — does form break down?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for seg in segments:
    sub = seg['data']
    if 'cadence' not in sub.columns: continue
    color = TERRAIN_COLORS.get(seg['terrain'], DEFAULT_COLOR)
    label = seg['name'].split('–')[-1].strip()

    ddist = sub['dist_km'].diff() * 1000
    dalt  = sub['alt'].diff()
    slope_pct = (dalt / ddist.replace(0, np.nan) * 100).rolling(15, center=True).mean().clip(-50, 50)

    mask = slope_pct.notna() & sub['cadence'].notna()
    axes[0].scatter(slope_pct[mask], sub['cadence'][mask],
                    color=color, alpha=0.25, s=8)
    # Trend line per segment
    if mask.sum() > 10:
        z = np.polyfit(slope_pct[mask], sub['cadence'][mask], 1)
        xs = np.linspace(slope_pct[mask].min(), slope_pct[mask].max(), 50)
        axes[0].plot(xs, np.polyval(z, xs), color=color, lw=2, label=label)

    if 'step_length' in sub.columns:
        mask2 = slope_pct.notna() & sub['step_length'].notna()
        axes[1].scatter(slope_pct[mask2], sub['step_length'][mask2],
                        color=color, alpha=0.25, s=8)
        if mask2.sum() > 10:
            z2 = np.polyfit(slope_pct[mask2], sub['step_length'][mask2], 1)
            xs2 = np.linspace(slope_pct[mask2].min(), slope_pct[mask2].max(), 50)
            axes[1].plot(xs2, np.polyval(z2, xs2), color=color, lw=2, label=label)

axes[0].set_xlabel('Slope [%]'); axes[0].set_ylabel('Cadence [spm]')
axes[0].set_title('Cadence vs slope — trend per segment')
axes[0].legend(fontsize=7, frameon=False); axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Slope [%]'); axes[1].set_ylabel('Step length [mm]')
axes[1].set_title('Step length vs slope — trend per segment')
axes[1].legend(fontsize=7, frameon=False); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('biomechanics_vs_slope.png', dpi=150, bbox_inches='tight'); plt.show()

### Speed variance within segments — technical difficulty proxy

In [ ]:
fig, ax = plt.subplots(figsize=(max(8, len(segments)*1.6), 4))

for i, seg in enumerate(segments):
    sub = seg['data']
    if 'speed' not in sub.columns: continue
    spd = sub['speed'].replace(0, np.nan).dropna()
    color = TERRAIN_COLORS.get(seg['terrain'], DEFAULT_COLOR)
    # Violin-style: box + scatter
    ax.boxplot(spd, positions=[i], widths=0.4, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='black', lw=2),
               whiskerprops=dict(color='grey'), capprops=dict(color='grey'),
               flierprops=dict(marker='.', color=color, alpha=0.3, ms=3))

names = [s['name'].split('–')[-1].strip() for s in segments]
ax.set_xticks(range(len(segments)))
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Speed [m/s]')
ax.set_title('Speed distribution per segment — wider box = more stop/go or technical terrain')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('speed_variance.png', dpi=150, bbox_inches='tight'); plt.show()

---
# ─── RACE-DAY READINESS DASHBOARD ───────────────────────────────
---

### Full race-day dashboard
Edit `TARGET_FINISH_MIN` and `HR_MAX` then run. Everything is derived from your segments.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────
TARGET_FINISH_MIN = 90.0    # goal finish time
HR_MAX            = 185     # your max HR (used for zone calculation)
HR_ZONES          = {       # as fraction of HR_MAX
    'Z1 Easy':       (0.50, 0.60),
    'Z2 Aerobic':    (0.60, 0.70),
    'Z3 Tempo':      (0.70, 0.80),
    'Z4 Threshold':  (0.80, 0.90),
    'Z5 Max':        (0.90, 1.00),
}
ZONE_COLORS = ['#4daf4a','#377eb8','#ff7f00','#e41a1c','#984ea3']
# ──────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#0d1117')
gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

DARK = '#0d1117'; CARD = '#161b22'; TEXT = '#e6edf3'; MUTED = '#8b949e'
ACCENT = '#f78166'

def dark_ax(ax):
    ax.set_facecolor(CARD)
    for spine in ax.spines.values(): spine.set_color('#30363d')
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    ax.title.set_color(TEXT)
    ax.grid(color='#21262d', alpha=0.8)
    return ax

# ── (0,0–2) Elevation + race plan overlay ──────────────────────
ax_main = dark_ax(fig.add_subplot(gs[0, :]))
offset = 0.0
for seg in segments:
    sub  = seg['data']
    color = TERRAIN_COLORS.get(seg['terrain'], DEFAULT_COLOR)
    x = sub['rel_dist'] + offset
    ax_main.fill_between(x, sub['alt'], sub['alt'].min() - 20, alpha=0.18, color=color)
    ax_main.plot(x, sub['alt'], color=color, lw=2.5)
    # Target pace annotation
    tp = next((r['target_pace'] for _, r in plan_df.iterrows()
                if seg['name'].split('–')[-1].strip() in r['segment']), None)
    if tp:
        mid = len(sub)//2
        ax_main.text(x.iloc[mid], sub['alt'].max() + 8,
                     f"{tp:.1f} min/km", color=color, fontsize=7.5,
                     ha='center', fontweight='bold')
    ax_main.axvline(offset, color='#30363d', lw=1, ls='--')
    offset += seg['dist_km']
ax_main.set_xlabel('Cumulative distance [km]'); ax_main.set_ylabel('Altitude [m]')
ax_main.set_title(f'Race profile with target paces — goal: {TARGET_FINISH_MIN:.0f} min', fontsize=11)

# ── (1,0) Pace plan bar ────────────────────────────────────────
ax1 = dark_ax(fig.add_subplot(gs[1, 0]))
if not plan_df.empty:
    xp = np.arange(len(plan_df))
    bar_colors = ['#f78166' if d < 0 else '#3fb950' for d in plan_df['delta_pct']]
    ax1.bar(xp, plan_df['target_pace'], color=bar_colors, alpha=0.85, edgecolor='none')
    ax1.set_xticks(xp)
    ax1.set_xticklabels(plan_df['segment'], rotation=30, ha='right', fontsize=7)
    ax1.invert_yaxis(); ax1.set_ylabel('Target pace [min/km]')
    ax1.set_title('Segment target paces')

# ── (1,1) Cumulative time projection ──────────────────────────
ax2 = dark_ax(fig.add_subplot(gs[1, 1]))
if not plan_df.empty:
    ax2.step(range(len(plan_df)+1),
             [0] + list(plan_df['cumulative_time']),
             color=ACCENT, lw=2.5, where='post')
    ax2.fill_between(range(len(plan_df)+1),
                     [0] + list(plan_df['cumulative_time']),
                     alpha=0.15, color=ACCENT, step='post')
    ax2.axhline(TARGET_FINISH_MIN, color='#3fb950', lw=1.5, ls='--', label=f'Goal {TARGET_FINISH_MIN:.0f} min')
    ax2.set_xticks(range(len(plan_df)+1))
    ax2.set_xticklabels(['Start'] + list(plan_df['segment']), rotation=30, ha='right', fontsize=7)
    ax2.set_ylabel('Elapsed time [min]'); ax2.set_title('Cumulative time projection')
    ax2.legend(fontsize=8, facecolor=CARD, edgecolor='none', labelcolor=TEXT)

# ── (1,2) HR zone distribution across whole run ───────────────
ax3 = dark_ax(fig.add_subplot(gs[1, 2]))
all_hr = np.concatenate([s['data']['hr'].dropna().values for s in segments if 'hr' in s['data'].columns])
zone_times = []
for (lo, hi), zcolor in zip(HR_ZONES.values(), ZONE_COLORS):
    count = np.sum((all_hr >= lo * HR_MAX) & (all_hr < hi * HR_MAX))
    zone_times.append(count)
wedges, texts, autotexts = ax3.pie(
    zone_times, labels=list(HR_ZONES.keys()), colors=ZONE_COLORS,
    autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(edgecolor=DARK, linewidth=1.5))
for t in texts: t.set_color(MUTED); t.set_fontsize(8)
for t in autotexts: t.set_color(TEXT); t.set_fontsize(8)
ax3.set_title('HR zone distribution (training run)')

# ── (2,0) D+/km difficulty bar ────────────────────────────────
ax4 = dark_ax(fig.add_subplot(gs[2, 0]))
seg_names_s = [s['name'].split('–')[-1].strip() for s in segments]
dplus_km    = [s['elev_gain'] / s['dist_km'] if s['dist_km'] > 0 else 0 for s in segments]
seg_colors  = [TERRAIN_COLORS.get(s['terrain'], DEFAULT_COLOR) for s in segments]
ax4.barh(seg_names_s, dplus_km, color=seg_colors, alpha=0.85, edgecolor='none')
ax4.set_xlabel('D+ per km [m/km]'); ax4.set_title('Segment difficulty (D+/km)')
ax4.invert_yaxis()

# ── (2,1) GAP vs actual pace ──────────────────────────────────
ax5 = dark_ax(fig.add_subplot(gs[2, 1]))
gap_v   = [s.get('gap_median', np.nan) for s in segments]
pace_v  = [s.get('pace_median', np.nan) for s in segments]
ax5.scatter(gap_v, pace_v, c=seg_colors, s=80, zorder=5, edgecolors='white', linewidths=0.5)
if any(~np.isnan(gap_v)):
    lo = min(v for v in gap_v+pace_v if not np.isnan(v))
    hi = max(v for v in gap_v+pace_v if not np.isnan(v))
    ax5.plot([lo, hi], [lo, hi], color=MUTED, lw=1, ls='--', alpha=0.6, label='GAP = pace (flat)')
for name, g, p in zip(seg_names_s, gap_v, pace_v):
    if not np.isnan(g): ax5.annotate(name, (g, p), fontsize=6.5, color=MUTED,
                                     xytext=(4, 2), textcoords='offset points')
ax5.set_xlabel('GAP [min/km]'); ax5.set_ylabel('Actual pace [min/km]')
ax5.invert_xaxis(); ax5.invert_yaxis()
ax5.set_title('GAP vs actual pace'); ax5.legend(fontsize=7, facecolor=CARD, edgecolor='none', labelcolor=TEXT)

# ── (2,2) Key numbers ─────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
ax6.set_facecolor(CARD)
ax6.set_xlim(0,1); ax6.set_ylim(0,1); ax6.axis('off')
total_dist  = sum(s['dist_km'] for s in segments)
total_dplus = sum(s['elev_gain'] for s in segments)
total_dminus= sum(abs(s['elev_loss']) for s in segments)
avg_hr_all  = np.nanmean([s['avg_hr'] for s in segments])
kvs = [
    ('Total distance',  f"{total_dist:.2f} km"),
    ('Total D+',        f"{total_dplus:.0f} m"),
    ('Total D-',        f"{total_dminus:.0f} m"),
    ('Avg D+/km',       f"{total_dplus/total_dist:.0f} m/km"),
    ('Goal finish',     f"{TARGET_FINISH_MIN:.0f} min"),
    ('Required GAP',    f"{target_gap:.2f} min/km"),
    ('Avg HR (training)',f"{avg_hr_all:.0f} bpm"),
    ('Segments',        f"{len(segments)}"),
]
for j, (k, v) in enumerate(kvs):
    y = 0.92 - j * 0.115
    ax6.text(0.05, y, k, color=MUTED, fontsize=8.5, va='top')
    ax6.text(0.95, y, v, color=TEXT,  fontsize=9.5, va='top', ha='right', fontweight='bold')
    ax6.axhline(y - 0.035, color='#21262d', lw=0.8, xmin=0.03, xmax=0.97)
ax6.set_title('Race snapshot', color=TEXT, pad=6)
for spine in ax6.spines.values(): spine.set_color('#30363d')
ax6.title.set_color(TEXT)

fig.suptitle('RACE-DAY READINESS DASHBOARD', color=TEXT, fontsize=14,
             fontweight='bold', y=0.98)
plt.savefig('race_day_dashboard.png', dpi=150, bbox_inches='tight', facecolor=DARK)
plt.show()
print('Saved: race_day_dashboard.png')